# Sesión 2 · Ejercicio 2 — Control
**Objetivo:** condicionar un generador a la clase y producir muestras de
SU clase minoritaria. **Aquí nace su mini-proyecto.**
**Tiempo:** MÍNIMO 25 min · COMPLETO 60 min
**Produce:** §4 de su bitácora + **un archivo de muestras sintéticas que
se usa en la Sesión 4**
**Necesitas:** su ficha de datos de §1 (para saber cuál es su minoritaria)


In [ ]:
#### OBLIGATORIO #### — setup (idempotente: puede ejecutarla dos veces)
import os, sys
if not os.path.isdir("src"):
    if not os.path.isdir("IAA6_M13_Gen"):
        !git clone -q https://github.com/AdriannaGmz/IAA6_M13_Gen
    %cd IAA6_M13_Gen
!pip install -q -r requirements.txt
sys.path.insert(0, ".")
from src import datos, modelos, evaluar, graficas, rescate
MODO_GPU = rescate.hay_gpu()   # imprime "GPU disponible" o "Modo CPU"


In [ ]:
#### OBLIGATORIO #### — sus datos (o los de demostración)
# Si su proyecto es tabular o de series, cambie MODO y FUENTE:
# el resto del cuaderno es idéntico. Ésa es la gracia de la capa común.
MODO = "tabular"     # ← su modalidad
FUENTE = None        # ← su archivo, o None para la demostración
X, y, meta = datos.cargar(MODO, FUENTE)
datos.ficha(meta)


In [ ]:
#### OBLIGATORIO #### — un generador que escucha
# condicional=True: la clase entra al codificador y al decodificador.
# Es el CVAE de las diapositivas del Bloque 3 de ayer.
cvae = modelos.VAE(meta, dim_latente=16, condicional=True)
historial = cvae.entrenar(
    X, y, epocas=30,
    cb=lambda e, r: (e % 5 == 4) and print(
        f"  época {e + 1:2d}/30 · pérdida {r['total']:8.1f}"))
graficas.curva(historial, "CVAE")


In [ ]:
# ── CELDA DE RESCATE ────────────────────────────────────────
# ¿No entrenó el CVAE? Ejecute esto: entrega directamente las
# muestras sintéticas genéricas (del conjunto de demostración
# tabular) y puede continuar desde la celda de comparación.
contenido, figuras = rescate.cargar("s2_sintetico_generico")
X_sint = contenido["X_sint"]
idx_min = meta["nombres_clases"].index(meta["clase_minoritaria"])
print(f"Recuperadas {len(X_sint)} muestras sintéticas genéricas "
      f"de la clase {contenido['clase']!r}.")


In [ ]:
#### OBLIGATORIO #### — pedir exactamente lo que falta
# COMPLETAR: genere 200 muestras condicionadas a SU clase minoritaria.
# Pista 1: el índice de la minoritaria es la posición de
#          meta["clase_minoritaria"] dentro de meta["nombres_clases"].
# Pista 2: cvae.muestrear acepta (n, y=indice_de_clase).
idx_min = ...
X_sint = ...
print(f"200 muestras sintéticas de la clase {meta['clase_minoritaria']!r}")


In [ ]:
#### OBLIGATORIO #### — cara a cara: lo real escaso vs lo sintético
# COMPLETAR: seleccione las muestras REALES de su clase minoritaria
# para compararlas con las sintéticas.
# Pista: una máscara booleana sobre las etiquetas: y == idx_min
mascara = ...
fig = graficas.comparar(X[mascara][:200], X_sint,
                        etiquetas=("real (escaso)", "sintético (CVAE)"))
fig.savefig("bitacora_s2e2_real_vs_sintetico.png", dpi=120)


In [ ]:
#### OBLIGATORIO #### — ⚠️ guardar el archivo del mini-proyecto
import torch
# COMPLETAR: guarde X_sint (en CPU) con torch.save en "sinteticos_s2.pt"
# Pista: torch.save({"X_sint": X_sint.cpu(), ...}, "sinteticos_s2.pt")
...
print("⚠️ DESCÁRGUELO Y CONSÉRVELO: en la Sesión 4 este archivo")
print("   compite contra un modelo de difusión, y el TSTR decide.")


### Observación — complete antes de cerrar

- ¿Las muestras sintéticas se ven **verosímiles**? ___
- ¿Son **variadas entre sí, o casi todas iguales**? ___
- (Si son casi iguales: ¿de qué fenómeno de hoy es pariente eso?) ___


In [ ]:
#### OBLIGATORIO #### — artefacto para la bitácora
print("Copie este bloque en la sección §4 de su bitácora y adjunte")
print("bitacora_s2e2_real_vs_sintetico.png:\n")
print(f"- Modalidad: {meta['modalidad']} · fuente: {meta['fuente']}")
print(f"- Clase generada: {meta['clase_minoritaria']} (200 muestras)")
print("- Archivo sintético guardado en: sinteticos_s2.pt (descargado)")
print("- ¿Verosímiles?: <...>  · ¿Variadas?: <...>")


### EXTENSIÓN (equipos rápidos)
1. Genere 50 muestras de CADA clase y compare las nubes con
   `datos.vistazo`. ¿Qué clase le sale mejor al CVAE? ¿Coincide con la
   que tiene más datos reales?
2. Si trajo datos propios: repita el cuaderno completo sobre ellos
   (cambie MODO y FUENTE arriba; todo lo demás es idéntico).


In [ ]:
#### EXTENSIÓN ####
import torch
por_clase = [cvae.muestrear(50, y=i) for i in range(meta["n_clases"])]
X_todo = torch.cat(por_clase)
y_todo = torch.arange(meta["n_clases"]).repeat_interleave(50)
datos.vistazo(X_todo, y_todo, meta)
